**Step 1-Connecting With Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content

Mounted at /content/drive
/content


**Step 2-Cloning The Repository**

In [ ]:
!git clone https://github.com/WongKinYiu/yolov9.git
!pip install -r requirements.txt
!pip install "albumentations<2"
%cd yolov9


Cloning into 'yolov9'...
remote: Enumerating objects: 781, done.
remote: Total 781 (delta 0), reused 0 (delta 0), pack-reused 781 (from 1)
Receiving objects: 100% (781/781), 3.27 MiB | 20.30 MiB/s, done.
Resolving deltas: 100% (330/330), done.
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.9/274.9 kB 6.1 MB/s eta 0:00:00
  Attempting uninstall: albucore
    Found existing installation: albucore 0.0.24
    Uninstalling albucore-0.0.24:
      Successfully uninstalled albucore-0.0.24
  Attempting uninstall: albumentations
    Found existing installation: albumentations 2.0.8
    Uninstalling albumentations-2.0.8:
      Successfully uninstalled albumentations-2.0.8
/content/yolov9


**Step 3-Splitting Dataset and Configuring Data File (data.yaml)**

In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split

path = '/content/drive/MyDrive/Project3'

files = os.listdir(f'{path}/images')

train, temp = train_test_split(files, test_size=0.3, random_state=42)
val, test = train_test_split(temp, test_size=0.5, random_state=42)

for folder in ['train', 'validation', 'test']:
    os.makedirs(f'{path}/{folder}/images', exist_ok=True)
    os.makedirs(f'{path}/{folder}/labels', exist_ok=True)

def copy_files(files, folder):
    for f in files:
        shutil.copy(f'{path}/images/{f}', f'{path}/{folder}/images/{f}')

        label = f.replace('.jpg', '.txt')
        shutil.copy(f'{path}/labels/{label}', f'{path}/{folder}/labels/{label}')

copy_files(train, 'train')
copy_files(val, 'validation')
copy_files(test, 'test')

# Create data.yaml
yaml_content = f"""
train: {path}/train/images
val: {path}/validation/images
test: {path}/test/images

nc: 2
names: ['cylinder','shock_absorber']
"""

with open('/content/yolov9/data.yaml', 'w') as f:
    f.write(yaml_content)


**Step 4- Downloading Yolov9 Model Weights**

In [ ]:
!wget -P /content/yolov9 https://github.com/WongKinYiu/yolov9/releases/download/v0.1/yolov9-c.pt

--2026-05-09 17:33:27--  https://github.com/WongKinYiu/yolov9/releases/download/v0.1/yolov9-c.pt
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/759338070/c8ca43f2-0d2d-4aa3-a074-426505bfbfb1?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-05-09T18%3A18%3A08Z&rscd=attachment%3B+filename%3Dyolov9-c.pt&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-05-09T17%3A18%3A06Z&ske=2026-05-09T18%3A18%3A08Z&sks=b&skv=2018-11-09&sig=Nt%2B3qJXM4VZRReyl11EhLQD%2FCXxLUsIVOYqr8gQ6kss%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3ODM0OTgwNywibmJmIjoxNzc4MzQ4MDA3LCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdGlvbi5ibG9iLmNvcm

### **Step5-Installing Required Libraries**

In [ ]:
import torch
import numpy as np
torch.serialization.add_safe_globals([np.core.multiarray._reconstruct])

/tmp/ipykernel_15692/959607124.py:3: DeprecationWarning: numpy.core is deprecated and has been renamed to numpy._core. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.multiarray.
  torch.serialization.add_safe_globals([np.core.multiarray._reconstruct])


In [ ]:
!pip install torch==2.7.0 torchvision==0.22.0 torchaudio==2.7.0

### **Step6-Model Training**

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

!python train_dual.py \
--workers 2 \
--device 0 \
--batch 4 \
--data /content/yolov9/data.yaml \
--img 320 \
--cfg models/detect/yolov9-c.yaml \
--weights yolov9-c.pt \
--name yolov9_hazwaste \
--hyp data/hyps/hyp.scratch-high.yaml \
--epochs 30

2026-05-09 09:58:45.284251: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice: (30 second timeout) 
wandb: WARNING W&B disabled due to login timeout.
wandb: ERROR Error while calling W&B API: api key too short (<Response [401]>)
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
train_dual: weights=yolov9-c.pt, cfg=models/detect/yolov9-c.yaml, data=/content/yolov9/data.yaml, hyp=data/hyps/hyp.scratch-high.yaml, epochs=30, batch_size=4, imgsz=320, rect=False, resume=False, nosave=False, noval=False, noautoanchor=False, noplots=False, evolve=None, bucket=, cache=None, image_weights=False, de

### **Step 7-Model Testing**

In [ ]:
%cd /content/yolov9

!python detect.py \
--weights runs/train/yolov9_hazwaste2/weights/best.pt \
--source /content/drive/MyDrive/Project3/test/images \
--img 320 \
--conf 0.25 \
--device 0

/content/yolov9
detect: weights=['runs/train/yolov9_hazwaste2/weights/best.pt'], source=/content/drive/MyDrive/Project3/test/images, data=data/coco128.yaml, imgsz=[320, 320], conf_thres=0.25, iou_thres=0.45, max_det=1000, device=0, view_img=False, save_txt=False, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=runs/detect, name=exp, exist_ok=False, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLO 🚀 v0.1-104-g5b1ea9a Python-3.12.13 torch-2.7.0+cu126 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
yolov9-c summary: 604 layers, 50700588 parameters, 0 gradients
image 1/84 /content/drive/MyDrive/Project3/test/images/Screenshot-2025-02-09-215322_png_png.rf.3d2af86c76fd3ad8fa83607507dc644a.jpg: 320x320 3 cylinders, 44.3ms
image 2/84 /content/drive/MyDrive/Project3/test/images/Screenshot-2025-02-09-215322_png_png.rf.4168f6737dd661440462db09723b2e53.jpg: 320x320 3

### **Step8-Saving model weights & output**

In [ ]:
!cp /content/yolov9/runs/train/yolov9_hazwaste2/weights/best.pt /content/drive/MyDrive/

!cp /content/yolov9/runs/train/yolov9_hazwaste2/weights/last.pt /content/drive/MyDrive/

In [ ]:
!cp -r /content/yolov9/runs/detect/exp5 /content/drive/MyDrive/

In [ ]:
!cp -r /content/yolov9 /content/drive/MyDrive/

## **Step9-Model Evaluation**

In [ ]:
%cd /content/yolov9

!python val_dual.py \
--weights /content/drive/MyDrive/Project3/best.pt \
--data /content/yolov9/data.yaml \
--img 320 \
--batch 4 \
--conf 0.001 \
--iou 0.65 \
--device 0 \
--task val

/content/yolov9
val_dual: data=/content/yolov9/data.yaml, weights=['/content/drive/MyDrive/Project3/best.pt'], batch_size=4, imgsz=320, conf_thres=0.001, iou_thres=0.65, max_det=300, task=val, device=0, workers=8, single_cls=False, augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, save_json=False, project=runs/val, name=exp, exist_ok=False, half=False, dnn=False, min_items=0
YOLO 🚀 v0.1-104-g5b1ea9a Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
yolov9-c summary: 604 layers, 50700588 parameters, 0 gradients
100% 755k/755k [00:00<00:00, 175MB/s]
val: Scanning /content/drive/MyDrive/Project3/validation/labels.cache... 83 images, 0 backgrounds, 0 corrupt: 100% 83/83 [00:00<?, ?it/s]
                 Class     Images  Instances          P          R      mAP50   mAP50-95:   5% 1/21 [00:01<00:28,  1.45s/it]Exception in thread Thread-3 (plot_images):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py",